# Simple Trajectory Profiler (Text Mode)

Minimal notebook to profile /conversations/{id}/messages using the API's built-in timings.

Server-side toggles you may want set before running:
- `TEXT_CONTEXT_MAX_MESSAGES=20`
- `TEXT_LLM_MAX_TOKENS=300`
- `MEMORY_RETRIEVAL_ENABLED=true|false`
- `MEMORY_RETRIEVAL_TIMEOUT_MS=600`

Client env used here:
- `EM_API_BASE` (default: http://localhost:8100)
- `EM_API_PREFIX` (optional, e.g., /api)
- `EM_COMPANION_ID` (required)
- `EM_LLM_PROVIDER` (default: openai-gpt4o-mini)
- `EM_TEMPERATURE` (default: 0.7)


In [ ]:
import json
import os
import time
from typing import Any, Dict, List

import httpx
import numpy as np
import pandas as pd

BASE_URL = os.getenv("EM_API_BASE", "http://localhost:8100").rstrip("/")
API_PREFIX = (os.getenv("EM_API_PREFIX", "") or "").strip("/")
BASE = f"{BASE_URL}/{API_PREFIX}" if API_PREFIX else BASE_URL
COMPANION_ID = "ece1da13-92b3-43f2-bb67-8b1fbc527afa"
LLM_PROVIDER = os.getenv("EM_LLM_PROVIDER", "openai-gpt4o-mini")
TEMPERATURE = float(os.getenv("EM_TEMPERATURE", "0.7"))
TIMEOUT_S = float(os.getenv("EM_HTTP_TIMEOUT", "30"))
HEADERS = {"Content-Type": "application/json", "X-EM-Debug-Timing": "1"}

assert COMPANION_ID, "Set EM_COMPANION_ID to a valid companion UUID."
client = httpx.Client(timeout=TIMEOUT_S, follow_redirects=True)


def api_url(path: str) -> str:
    if not path.startswith("/"):
        path = "/" + path
    return f"{BASE}{path}"


def create_conversation(companion_id: str) -> str:
    url = api_url("/conversations/")  # trailing slash avoids 307
    r = client.post(
        url, headers={"Content-Type": "application/json"}, json={"companion_id": companion_id}
    )
    r.raise_for_status()
    return r.json()["id"]


def send_text(
    conversation_id: str, content: str, *, provider: str, temperature: float
) -> Dict[str, Any]:
    url = api_url(f"/conversations/{conversation_id}/messages")
    payload = {
        "content": content,
        "system_prompt": "Use server-configured prompt",
        "llm_provider": provider,
        "temperature": temperature,
    }
    t0 = time.perf_counter()
    r = client.post(url, headers=HEADERS, json=payload)
    t1 = time.perf_counter()
    r.raise_for_status()
    data = r.json()
    timings = data.get("timings") or {}
    timings = {k: float(v) for k, v in timings.items()} if isinstance(timings, dict) else {}
    timings["total_client_ms"] = (t1 - t0) * 1000.0
    return {"data": data, "timings": timings}


def p50(xs: List[float]) -> float:
    return float(np.percentile(xs, 50)) if xs else float("nan")


def p95(xs: List[float]) -> float:
    return float(np.percentile(xs, 95)) if xs else float("nan")

In [ ]:
for i in range(N):
    print(df_search.iloc[i].to_dict())

In [ ]:
for i in range(N):
    print(df_search.iloc[i].to_dict())

In [ ]:
for i in range(N):
    print(df_search.iloc[i].to_dict())

## Single Turn (raw output + timings)

In [ ]:
conv_id = create_conversation(COMPANION_ID)
resp = send_text(conv_id, "Latency probe turn", provider=LLM_PROVIDER, temperature=TEMPERATURE)
print("Conversation ID:", conv_id)
print("Response keys:", list(resp["data"].keys()))
print("Timings keys:", sorted(resp["timings"].keys()))
print("Timings (ms):", json.dumps({k: round(v, 2) for k, v in resp["timings"].items()}, indent=2))
# Uncomment to inspect the full response
# print(json.dumps(resp['data'], indent=2)[:1200])

## Multi-turn Script (summary + per-turn table)

In [ ]:
script = [
    "Hi there!",
    "For future reference, my name is Alex and I love hiking.",
    "Cool. What did I tell you about my name?",
    "Plan a weekend: I usually prefer outdoor activities.",
    "Thanks! That works.",
]

rows: List[Dict[str, Any]] = []
for i, u in enumerate(script, start=1):
    r = send_text(conv_id, u, provider=LLM_PROVIDER, temperature=TEMPERATURE)
    t = r["timings"]
    known = sum(
        [
            float(t.get("persist_user_ms", 0.0)),
            float(t.get("prompt_assemble_ms", 0.0)),
            float(t.get("retrieval_ms", 0.0)),
            float(t.get("llm_ms", 0.0)),
            float(t.get("persist_assistant_ms", 0.0)),
        ]
    )
    t["overhead_ms"] = float(t.get("total_client_ms", 0.0)) - known
    rows.append({"turn": i, **t})
    # print all timings (incl. LLM breakdown if present)
    print(
        f"Turn {i}: total={t.get('total_client_ms', float('nan')):.1f} ms, "
        f"llm={t.get('llm_ms', float('nan')):.1f} ms, "
        f"llm_assemble={t.get('llm_assemble_ms', float('nan')):.1f} ms, "
        f"llm_convert={t.get('llm_convert_ms', float('nan')):.1f} ms, "
        f"llm_client={t.get('llm_client_select_ms', float('nan')):.1f} ms, "
        f"llm_api={t.get('llm_api_ms', float('nan')):.1f} ms, "
        f"llm_extract={t.get('llm_extract_ms', float('nan')):.1f} ms, "
        f"retrieval={t.get('retrieval_ms', 0.0):.1f} ms, "
        f"history_msgs={t.get('history_msgs', 'na')}, truncated={t.get('history_truncated', 'na')}, "
        f"persist_user_ms={t.get('persist_user_ms', float('nan')):.1f} ms, "
        f"prompt_assemble_ms={t.get('prompt_assemble_ms', float('nan')):.1f} ms, "
        f"heuristic_ms={t.get('heuristic_ms', float('nan')):.1f} ms, "
        f"retrieval_embed_ms={t.get('retrieval_embed_ms', float('nan')):.1f} ms, "
        f"retrieval_db_ms={t.get('retrieval_db_ms', float('nan')):.1f} ms, "
        f"persist_assistant_ms={t.get('persist_assistant_ms', float('nan')):.1f} ms, "
        f"tok_in={int(t.get('llm_prompt_tokens', 0.0))}, tok_out={int(t.get('llm_output_tokens', 0.0))}, tok_total={int(t.get('llm_total_tokens', 0.0))}"
    )

df = pd.DataFrame(rows).fillna(0)
df

In [ ]:
def summarize(df: pd.DataFrame, cols: List[str]) -> Dict[str, Dict[str, float]]:
    out: Dict[str, Dict[str, float]] = {}
    for c in cols:
        if c not in df.columns:
            continue
        xs = df[c].astype(float).tolist()
        out[c] = {
            "mean": float(np.mean(xs)) if xs else float("nan"),
            "p50": p50(xs),
            "p95": p95(xs),
            "n": len(xs),
        }
    return out


time_cols = [
    "total_client_ms",
    "persist_user_ms",
    "db_history_ms",
    "effective_prompt_ms",
    "prompt_assemble_ms",
    "heuristic_ms",
    "retrieval_embed_ms",
    "retrieval_db_ms",
    "retrieval_ms",
    "llm_ms",
    "llm_assemble_ms",
    "llm_convert_ms",
    "llm_client_select_ms",
    "llm_api_ms",
    "llm_extract_ms",
    "persist_assistant_ms",
]
token_cols = ["llm_prompt_tokens", "llm_output_tokens", "llm_total_tokens"]
summary_time = summarize(df, time_cols)
print("Time Summary (p50/p95/mean in ms):")
for k in time_cols:
    if k in summary_time:
        v = summary_time[k]
        print(
            k,
            json.dumps(
                {"p50": round(v["p50"], 2), "p95": round(v["p95"], 2), "mean": round(v["mean"], 2)}
            ),
        )
print()
summary_tok = summarize(df, token_cols)
print("Token Summary (p50/p95/mean counts):")
for k in token_cols:
    if k in summary_tok:
        v = summary_tok[k]
        print(
            k,
            json.dumps(
                {"p50": round(v["p50"], 2), "p95": round(v["p95"], 2), "mean": round(v["mean"], 2)}
            ),
        )
pd.DataFrame({**summary_time, **summary_tok}).T.round(2)

In [ ]:
import os

import pandas as pd
import requests

API_BASE = os.getenv("API_BASE", "http://localhost:8100")
COMPANION_ID = "ece1da13-92b3-43f2-bb67-8b1fbc527afa"


def dev_token():
    try:
        r = requests.get(f"{API_BASE}/api/auth/dev-token", timeout=10)
        if r.ok:
            return r.json().get("token", "")
    except Exception:
        pass
    return ""


TOKEN = dev_token()  # or set a real JWT if auth is enabled
HDRS = {"Authorization": f"Bearer {TOKEN}"} if TOKEN else {}


import pandas as pd


def _check(r):
    if not r.ok:
        try:
            print("Error body:", r.json())
        except Exception:
            print("Error text:", r.text)
        r.raise_for_status()


def create_conversation(companion_id: str) -> str:
    r = requests.post(
        f"{API_BASE}/conversations/", json={"companion_id": companion_id}, headers=HDRS, timeout=20
    )
    _check(r)
    return r.json()["id"]


def seed_history(conversation_id: str, pairs=200, avg_chars=160):
    r = requests.post(
        f"{API_BASE}/conversations/{conversation_id}/seed",
        json={"pairs": pairs, "avg_chars": avg_chars},
        headers=HDRS,
        timeout=120,
    )
    _check(r)
    return r.json()


def send_text(conversation_id: str, content: str, *, provider: str, temperature: float):
    t0 = time.perf_counter()
    r = requests.post(
        f"{API_BASE}/conversations/{conversation_id}/messages",
        json={
            "content": content,
            "system_prompt": "ignore",
            "llm_provider": provider,
            "temperature": temperature,
        },
        headers=HDRS,
        timeout=120,
    )
    t1 = time.perf_counter()
    _check(r)
    data = r.json()
    tim = data.get("timings", {})
    tim["total_client_ms"] = (t1 - t0) * 1000.0
    return {"resp": data, "timings": tim}

In [ ]:
def create_conversation(companion_id: str) -> str:
    r = requests.post(
        f"{API_BASE}/conversations/", json={"companion_id": companion_id}, headers=HDRS, timeout=20
    )
    r.raise_for_status()
    return r.json()["id"]


conv_id = create_conversation(COMPANION_ID)
conv_id

In [ ]:
def seed_history(conversation_id: str, pairs=200, avg_chars=160):
    r = requests.post(
        f"{API_BASE}/conversations/{conversation_id}/seed",
        json={"pairs": pairs, "avg_chars": avg_chars},
        headers=HDRS,
        timeout=60,
    )
    r.raise_for_status()
    return r.json()


seed_history(conv_id, pairs=200, avg_chars=160)

In [ ]:
# Reuse your existing send_text(); if not present, define a minimal one:


def send_text(conversation_id: str, content: str, *, provider: str, temperature: float):
    t0 = time.perf_counter()
    r = requests.post(
        f"{API_BASE}/conversations/{conversation_id}/messages",
        json={
            "content": content,
            "system_prompt": "ignore",
            "llm_provider": provider,
            "temperature": temperature,
        },
        headers=HDRS,
        timeout=60,
    )
    t1 = time.perf_counter()
    r.raise_for_status()
    data = r.json()
    tim = data.get("timings", {})
    tim["total_client_ms"] = (t1 - t0) * 1000.0
    return {"resp": data, "timings": tim}


LLM_PROVIDER = "openai-gpt4o"  # or what you normally use
TEMPERATURE = 0.7

# Warmup (fills prompt + history caches)

_ = send_text(conv_id, "Warmup turn (ignore).", provider=LLM_PROVIDER, temperature=TEMPERATURE)

# Your script with seeded long history

script = [
    "Hi there!",
    "For future reference, my name is Alex and I love hiking.",
    "Cool. What did I tell you about my name?",
    "Plan a weekend: I usually prefer outdoor activities.",
    "Thanks! That works.",
]

rows = []
for i, u in enumerate(script, start=1):
    r = send_text(conv_id, u, provider=LLM_PROVIDER, temperature=TEMPERATURE)
    t = r["timings"]
    rows.append({"turn": i, **t})
    print(
        f"Turn {i}: total={t.get('total_client_ms', float('nan')):.1f} ms, "
        f"llm={t.get('llm_ms', float('nan')):.1f} ms, retrieval={t.get('retrieval_ms', 0.0):.1f} ms, "
        f"history_msgs={t.get('history_msgs', 'na')}, truncated={t.get('history_truncated', 'na')}, "
        f"persist_user_ms={t.get('persist_user_ms', float('nan')):.1f} ms, "
        f"prompt_assemble_ms={t.get('prompt_assemble_ms', float('nan')):.1f} ms, "
        f"heuristic_ms={t.get('heuristic_ms', float('nan')):.1f} ms, "
        f"retrieval_embed_ms={t.get('retrieval_embed_ms', float('nan')):.1f} ms, "
        f"retrieval_db_ms={t.get('retrieval_db_ms', float('nan')):.1f} ms, "
        f"persist_assistant_ms={t.get('persist_assistant_ms', float('nan')):.1f} ms"
    )

df = pd.DataFrame(rows).fillna(0)
df